In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# .env を読み込む
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

In [2]:
from langchain_community.document_loaders import GitLoader

def file_filter(p: str) -> bool:
    p = p.lower()
    return p.endswith(".md") or p.endswith(".mdx")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print("documents:", len(documents))
print("sample:", documents[0].metadata.get("source"), documents[0].page_content[:200])

documents: 28
sample: AGENTS.md # Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context



In [4]:
for document in documents:
    document.metadata["filename"] = document.metadata["source"]

In [17]:
import nest_asyncio
nest_asyncio.apply()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

# 1. モデル準備
generator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o", temperature=0)
)

embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings()
)

# 2. TestsetGenerator 初期化
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=embeddings
)

# 3. ドキュメント分割（←ここが重要）
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

# 4. テストセット生成（transformsは使わない）
testset = generator.generate_with_langchain_docs(
    documents=split_docs,
    testset_size=4,
)

# 5. DataFrame確認
test_df = testset.to_pandas()
print(test_df.head())

/var/folders/qy/38qlvhzx5vb0bwwrbdm4q7_80000gn/T/ipykernel_46753/1520661911.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(
/var/folders/qy/38qlvhzx5vb0bwwrbdm4q7_80000gn/T/ipykernel_46753/1520661911.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(


Applying SummaryExtractor:   0%|          | 0/85 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/98 [00:00<?, ?it/s]

Node 5b5d48a8-5ffb-4a13-b0a4-b7de1a01aa3a does not have a summary. Skipping filtering.
Node d5f26234-c741-4c4f-bd61-9da7fea2642b does not have a summary. Skipping filtering.
Node 2aa52ab1-606c-4364-b141-4577c31c93e5 does not have a summary. Skipping filtering.
Node 70abd11f-1dea-474f-b4d3-d90cc6d70c28 does not have a summary. Skipping filtering.
Node c4d11b12-2885-484e-a293-3d977ce4c124 does not have a summary. Skipping filtering.
Node add3a19c-fd81-4dec-af25-7cc470743391 does not have a summary. Skipping filtering.
Node 45b4eb48-230f-463a-9449-b1ef2574e37c does not have a summary. Skipping filtering.
Node 2b29654b-8259-4b69-ab4c-0f47afbe88bc does not have a summary. Skipping filtering.
Node cc71118e-5234-4f7a-9b0e-ab6183dd61c0 does not have a summary. Skipping filtering.
Node 46dac10f-64ae-4c63-bfeb-abfc5e02878d does not have a summary. Skipping filtering.
Node 8c51b5ce-8a42-4054-bb09-55decb8f14e5 does not have a summary. Skipping filtering.
Node 762205f0-e2de-4178-90cd-3278a1690055 d

Applying EmbeddingExtractor:   0%|          | 0/85 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

                                          user_input  \
0  What is the structure of the LangChain Python ...   
1  Could you provide a detailed explanation of th...   
2  What are the core development principles and d...   
3  Howw does the versioning of third-party provid...   
4  How does the actively maintained 'langchain_v1...   

                                  reference_contexts  \
0  [# Global development guidelines for the LangC...   
1  [```txt\nlangchain/\n├── libs/\n│   ├── core/ ...   
2  [<1-hop>\n\n#### Pull request guidelines\n\n- ...   
3  [<1-hop>\n\n(Each package contains its own `RE...   
4  [<1-hop>\n\n```txt\nlangchain/\n├── libs/\n│  ...   

                                           reference  \
0  The LangChain Python project is structured as ...   
1  The 'langchain_v1' package is an actively main...   
2  The core development principles for ensuring c...   
3  The versioning of third-party provider integra...   
4  The 'langchain_v1' package is an actively m

In [18]:
testset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the structure of the LangChain Python ...,[# Global development guidelines for the LangC...,The LangChain Python project is structured as ...,Software Engineer,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
1,Could you provide a detailed explanation of th...,[```txt\nlangchain/\n├── libs/\n│ ├── core/ ...,The 'langchain_v1' package is an actively main...,Software Development Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,What are the core development principles and d...,[<1-hop>\n\n#### Pull request guidelines\n\n- ...,The core development principles for ensuring c...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
3,Howw does the versioning of third-party provid...,[<1-hop>\n\n(Each package contains its own `RE...,The versioning of third-party provider integra...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,How does the actively maintained 'langchain_v1...,[<1-hop>\n\n```txt\nlangchain/\n├── libs/\n│ ...,The 'langchain_v1' package is an actively main...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
5,How can I install LangChain.ai using pip?,[<1-hop>\n\n# langchain-mistralai\n\n[![PyPI -...,You can install LangChain.ai by using the foll...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [19]:
from langsmith import Client

dataset_name = "agent-book"

client = Client()

if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)

dataset = client.create_dataset(dataset_name=dataset_name)

In [21]:
inputs = []
outputs = []
metadatas = []

for sample in testset.samples:
    eval_sample = sample.eval_sample

    inputs.append({
        "question": getattr(eval_sample, "user_input", None)
    })

    outputs.append({
        "contexts": getattr(eval_sample, "reference_contexts", None),
        "ground_truth": getattr(eval_sample, "reference", None),
    })

    metadatas.append({
        "synthesizer_name": getattr(sample, "synthesizer_name", None),
    })